# AutoDL 数据下载 setup notebook

**目标:** 把 OlmoEarth Plan B(4.67 TB)+ V-JEPA ckpt 下载到 AutoDL 数据盘 (`/root/autodl-tmp/staging/`),后续可选上传阿里云盘归档。

**前置条件:**
- AutoDL 无卡 CPU / 1卡 GPU 实例 + 5 TB 数据盘挂载到 `/root/autodl-tmp/`
- Python 3.x 内核

**执行顺序:**
1. §1 环境配置 + 测连通性(必做)
2. §2 小数据测试,验证 toolchain(强烈建议)
3. §3.1 **一键启动全量后台下载** ★ ← 主战场,自动重试 + nohup,关 notebook 不影响
4. §4 监控,随时跑
5. §5 (可选)阿里云盘归档
6. §6 切 GPU 实例训练

---
## 1. 环境配置

In [ ]:
# 1.1 装 huggingface_hub + Rust 加速器
# 新版 huggingface_hub CLI 是 `hf`(不再是 huggingface-cli)
!pip install -q -U huggingface_hub hf_transfer
!hf --version

In [ ]:
# 1.2 设环境变量(切到 hf-mirror.com 国内镜像 + 启用 Rust 下载器)
import os
from pathlib import Path

os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "1"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"

# 写入 ~/.bashrc 让后续新终端都生效
bashrc = Path.home() / ".bashrc"
existing = bashrc.read_text() if bashrc.exists() else ""
bashrc_lines = [
    "export HF_ENDPOINT=https://hf-mirror.com",
    "export HF_HUB_ENABLE_HF_TRANSFER=1",
    "export HF_HUB_DOWNLOAD_TIMEOUT=120",
]
with bashrc.open("a") as f:
    for line in bashrc_lines:
        if line not in existing:
            f.write(f"\n{line}\n")

for k in ["HF_ENDPOINT", "HF_HUB_ENABLE_HF_TRANSFER", "HF_HUB_DOWNLOAD_TIMEOUT"]:
    print(f"{k} = {os.environ[k]}")


In [ ]:
# 1.3 检查数据盘空间 + 创建 staging 目录
import shutil, os

DATA_ROOT = "/root/autodl-tmp"
STAGING = f"{DATA_ROOT}/staging"

total, used, free = shutil.disk_usage(DATA_ROOT)
print(f"数据盘: total={total/2**40:.2f} TB, used={used/2**40:.2f} TB, free={free/2**40:.2f} TB")

# Plan B 约 4.67 TB; hf download 会产生 cache/lock/临时文件,5 TB 很紧。
# 如果只先下 S2+S1+ckpt,4.8 TB 也够;全量建议 5.2 TB+。
assert free / 2**40 > 5.2, (
    f"剩余空间 {free/2**40:.1f} TB,全量下载余量偏小。"
    "建议扩容到 5.2 TB+,或先只下 S2+S1+ckpt。"
)

os.makedirs(STAGING, exist_ok=True)
print(f"Staging 目录就绪: {STAGING}")


In [ ]:
# 1.4 测速(-L 跟踪 redirect,否则只测到 302 跳转)
!curl -L -s -o /dev/null -w "HTTP %{http_code}  速度 %{speed_download} bytes/s  耗时 %{time_total}s\n" \
  --max-time 60 \
  https://hf-mirror.com/datasets/allenai/olmoearth_pretrain_dataset/resolve/main/10_cdl/0000.tar

---
## 2. 小数据集测试(~3 min)

先下 4 个小 folder 验证 toolchain,无误再上 §3 大头。

In [ ]:
# 2.1 测试下载:10_cdl(656 MB,~30 sec)
!HF_ENDPOINT=https://hf-mirror.com HF_HUB_ENABLE_HF_TRANSFER=1 \
  hf download allenai/olmoearth_pretrain_dataset \
    --repo-type dataset \
    --include "10_cdl/*" \
    --local-dir /root/autodl-tmp/staging

In [ ]:
# 2.2 检查下载结果
!ls -lh /root/autodl-tmp/staging/10_cdl/
!du -sh /root/autodl-tmp/staging/10_cdl/

In [ ]:
# 2.3 其他 3 个小 folder,逐个跑(hf 多 --include 有 bug)
# WorldCover 3.2 GB + WorldCereal 9.2 GB + SRTM 14 GB,~2-5 min
import os
for folder in ["10_srtm", "10_worldcereal", "10_worldcover"]:
    print(f"=== {folder} ===")
    os.system(
        f'HF_ENDPOINT=https://hf-mirror.com HF_HUB_ENABLE_HF_TRANSFER=1 '
        f'hf download allenai/olmoearth_pretrain_dataset '
        f'--repo-type dataset --include "{folder}/*" '
        f'--local-dir /root/autodl-tmp/staging'
    )

In [ ]:
# 2.4 V-JEPA 2.1 ViT-L ckpt(700 MB) — 已存在则跳过,缺失才补下载
!mkdir -p /root/autodl-tmp/staging/vjepa
!if [ -s /root/autodl-tmp/staging/vjepa/vjepa2_1_vitl_dist_vitG_384.pt ]; then     echo "ckpt 已存在,跳过下载";   else     wget -c --tries=10 --timeout=120       -O /root/autodl-tmp/staging/vjepa/vjepa2_1_vitl_dist_vitG_384.pt       https://dl.fbaipublicfiles.com/vjepa2/vjepa2_1_vitl_dist_vitG_384.pt;   fi
!ls -lh /root/autodl-tmp/staging/vjepa/


In [ ]:
# 2.5 小数据全部验证
!du -sh /root/autodl-tmp/staging/*

---
## 3. 启动全量下载(一键后台,3-5 天自动跑完)

§3.1 是**主战场**:bash 脚本顺序下所有 folder + 每个最多重试 200 次,nohup 后台跑。**关 notebook / 关浏览器都不影响**。

§3.2 可选:`aria2c + hfd` 多线程,白天补充速度。

In [ ]:
# 3.1 ★ 一键启动:全量下载,hfd 多线程,自动重试,后台 2-3 天跑完
import os, time, subprocess

# 关闭代理;AutoDL 走 hf-mirror,不要让代理劫持 aria2/hfd
for k in ("https_proxy", "http_proxy", "all_proxy", "HTTPS_PROXY", "HTTP_PROXY", "ALL_PROXY"):
    os.environ.pop(k, None)

# 杀旧脚本(hf download 单流太慢;你当前结果约 1-2 MB/s,hfd 测到约 24 MB/s)
subprocess.run("pkill -f download_all.sh", shell=True)
time.sleep(2)

# 确保 hfd + aria2c 已装
subprocess.run(
    "which aria2c || (apt-get update -qq && apt-get install -y aria2 2>&1 | tail -1)",
    shell=True,
)
subprocess.run(
    "which hfd || (wget -q https://hf-mirror.com/hfd/hfd.sh -O /usr/local/bin/hfd && chmod +x /usr/local/bin/hfd)",
    shell=True,
)

# 写新脚本:hfd 多线程 + 自动 resume。失败会写 FAILED 并以非 0 退出。
script = r"""#!/bin/bash
set -u
LOG=/root/download.log
FAILED=0

echo "============================================" >> $LOG
echo "=== $(date) START hfd download ===" >> $LOG
echo "============================================" >> $LOG

export HF_ENDPOINT=https://hf-mirror.com
export STAGING=/root/autodl-tmp/staging
mkdir -p $STAGING

download_folder() {
  local folder=$1
  local jobs=$2
  local conns=$3
  echo "[$(date)] [$folder] starting (hfd -x $conns -j $jobs)" | tee -a $LOG
  for attempt in $(seq 1 200); do
    echo "[$(date)] [$folder] attempt $attempt" >> $LOG
    hfd allenai/olmoearth_pretrain_dataset \
      --dataset \
      --include "$folder/*" \
      --local-dir $STAGING \
      -x $conns -j $jobs >> $LOG 2>&1
    rc=$?
    if [ $rc -eq 0 ]; then
      echo "[$(date)] [$folder] DONE after $attempt attempts" | tee -a $LOG
      return 0
    fi
    echo "[$(date)] [$folder] failed (rc=$rc), retry in 60s..." >> $LOG
    sleep 60
  done
  echo "[$(date)] [$folder] FAILED after 200 attempts" | tee -a $LOG
  return 1
}

run_folder() {
  local folder=$1
  local jobs=$2
  local conns=$3
  if ! download_folder "$folder" "$jobs" "$conns"; then
    FAILED=1
  fi
}

# 1. 小 folder:单文件,不用太多并发;已存在会自动跳过/复用
run_folder 10_cdl 1 10
run_folder 10_worldcover 1 10
run_folder 10_worldcereal 1 10
run_folder 10_srtm 1 10

# 2. V-JEPA ckpt:你已经下好了就跳过;缺失才补
CKPT=$STAGING/vjepa/vjepa2_1_vitl_dist_vitG_384.pt
echo "[$(date)] [vjepa] checking..." | tee -a $LOG
mkdir -p $STAGING/vjepa
if [ -s $CKPT ]; then
  echo "[$(date)] [vjepa] exists, skip: $(du -h $CKPT | cut -f1)" | tee -a $LOG
else
  CKPT_OK=0
  for attempt in $(seq 1 50); do
    wget -c --tries=10 --timeout=120 -q \
      -O $CKPT \
      https://dl.fbaipublicfiles.com/vjepa2/vjepa2_1_vitl_dist_vitG_384.pt >> $LOG 2>&1
    if [ $? -eq 0 ] && [ -s $CKPT ]; then
      echo "[$(date)] [vjepa] DONE" | tee -a $LOG
      CKPT_OK=1
      break
    fi
    echo "[$(date)] [vjepa] failed, retry in 60s..." >> $LOG
    sleep 60
  done
  if [ $CKPT_OK -ne 1 ]; then
    echo "[$(date)] [vjepa] FAILED" | tee -a $LOG
    FAILED=1
  fi
fi

# 3. 大头:S2/S1 多文件并发。
#    你实测 hfd -x 10 约 24 MB/s;这里 -j 4 表示 4 个 shard 并行。
#    若速度下降或 mirror 限流,把 jobs 改成 1 或 2。
run_folder 10_sentinel2_l2a_monthly 4 10
run_folder 10_sentinel1_monthly 4 10

# 4. 结束前检查 incomplete/lock/tmp;有残留不标 complete
BAD_FILES=$(find $STAGING -type f \( -name "*.incomplete" -o -name "*.lock" -o -name "*.tmp" \) 2>/dev/null | wc -l)
if [ "$BAD_FILES" -gt 0 ]; then
  echo "[$(date)] Found $BAD_FILES incomplete/lock/tmp files" | tee -a $LOG
  FAILED=1
fi

echo "============================================" >> $LOG
if [ $FAILED -eq 0 ]; then
  echo "=== $(date) ALL DOWNLOADS COMPLETE ===" | tee -a $LOG
else
  echo "=== $(date) DOWNLOADS FINISHED WITH FAILURES ===" | tee -a $LOG
fi
echo "============================================" >> $LOG
exit $FAILED
"""

with open("/root/download_all.sh", "w") as f:
    f.write(script)
os.chmod("/root/download_all.sh", 0o755)

# 启动新版后台
subprocess.Popen(
    "nohup bash /root/download_all.sh > /dev/null 2>&1 &",
    shell=True,
    preexec_fn=os.setpgrp,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL,
)
time.sleep(3)

print("✓ hfd 版本后台下载已启动")
result = subprocess.run("ps aux | grep download_all.sh | grep -v grep", shell=True, capture_output=True, text=True)
print(result.stdout if result.stdout.strip() else "⚠️ 没检测到进程,看 /root/download.log")
print("日志: /root/download.log")


In [ ]:
# 3.2 (可选)用 hfd 多线程接管某个 folder
# hfd = hf-mirror 官方 wrapper,基于 aria2c,16 连接 + 自动 resume
# 白天觉得 §3.1 慢可以暂停它,用 hfd 接管 S2

# 1. 装依赖
!which aria2c || (apt-get update -qq && apt-get install -y aria2 2>&1 | tail -3)
!wget -q https://hf-mirror.com/hfd/hfd.sh -O /usr/local/bin/hfd
!chmod +x /usr/local/bin/hfd
!hfd 2>&1 | head -3

# 2. 用例(取消注释跑)
# 暂停主下载:
# !pkill -f download_all.sh

# 用 hfd 跑某个 folder(16 连接 × 4 并行)
# !HF_ENDPOINT=https://hf-mirror.com hfd allenai/olmoearth_pretrain_dataset \
#   --dataset --include "10_sentinel2_l2a_monthly/*" \
#   --local-dir /root/autodl-tmp/staging \
#   -x 16 -j 4

# 跑完恢复主脚本:
# !nohup bash /root/download_all.sh > /dev/null 2>&1 &

print("hfd 准备好,用法见上面注释")

---
## 4. 监控 + 验证

随时跑,看进度。**§3.1 后台脚本不依赖 notebook,关浏览器照样跑。**

In [ ]:
# 4.1 ★ 全方位状态(常跑这个)
import os, subprocess

STAGING = "/root/autodl-tmp/staging"

def shell(cmd):
    return subprocess.run(cmd, shell=True, capture_output=True, text=True).stdout.strip()

# 1. 后台脚本是否还在跑
print("=== 1. 后台进程 ===")
ps = shell("ps aux | grep -E 'download_all|huggingface_hub|hf_transfer|hfd|aria2c' | grep -v grep")
if ps:
    print("在跑:")
    print(ps)
else:
    print("⚠️ 没有下载进程,可能已完成或挂了")
    print("   重启: !nohup bash /root/download_all.sh > /dev/null 2>&1 &")

# 2. 每个 folder 大小 + tar 数
print("\n=== 2. 各 folder 进度 ===")
expected = {
    "10_sentinel2_l2a_monthly": 2.60 * 2**40,
    "10_sentinel1_monthly":     2.04 * 2**40,
    "10_srtm":                  14.1 * 2**30,
    "10_worldcereal":           9.2  * 2**30,
    "10_worldcover":            3.2  * 2**30,
    "10_cdl":                   656  * 2**20,
    "vjepa":                    700  * 2**20,
}
def fmt(b):
    if b >= 2**40: return f"{b/2**40:.2f} TB"
    if b >= 2**30: return f"{b/2**30:.1f} GB"
    if b >= 2**20: return f"{b/2**20:.0f} MB"
    return f"{b} B"

total_actual = total_expected = 0
print(f"{'Folder':<32} {'Size':>10}  {'Expected':>10}  {'TARs':>6}  Status")
print("-" * 82)
for folder, exp in expected.items():
    path = f"{STAGING}/{folder}"
    actual = sum(os.path.getsize(os.path.join(r, f))
                 for r, _, fs in os.walk(path) for f in fs) if os.path.exists(path) else 0
    tar_count = int(shell(f"find {path} -type f -name '*.tar' 2>/dev/null | wc -l") or 0) if os.path.exists(path) else 0
    total_actual += actual
    total_expected += exp
    if actual == 0:
        status = "未开始"
    elif actual / exp > 0.95:
        status = "大小OK ✓"
    else:
        status = f"{actual/exp:.0%}"
    print(f"{folder:<32} {fmt(actual):>10}  {fmt(exp):>10}  {tar_count:>6}  {status}")
print("-" * 82)
print(f"{'总计':<32} {fmt(total_actual):>10}  {fmt(total_expected):>10}          {total_actual/total_expected:.0%}")

# 3. incomplete / lock / tmp 文件
print("\n=== 3. 未完成文件 ===")
bad = shell(f"find {STAGING} -type f \\( -name '*.incomplete' -o -name '*.lock' -o -name '*.tmp' \\) 2>/dev/null | head -20")
if bad:
    print("发现未完成/锁文件(前20个):")
    print(bad)
else:
    print("未发现 incomplete/lock/tmp 文件 ✓")

# 4. 数据盘剩余
print("\n=== 4. 数据盘 ===")
print(shell("df -h /root/autodl-tmp | tail -2"))

# 5. 最近日志
print("\n=== 5. 最近日志(最后 20 行) ===")
print(shell("tail -20 /root/download.log 2>/dev/null"))

# 6. 可选:用 HF API 对比 shard 数(联网,失败不影响本地检查)
print("\n=== 6. HF shard 数参考(可选) ===")
try:
    from huggingface_hub import HfApi
    api = HfApi(endpoint="https://hf-mirror.com")
    files = api.list_repo_files("allenai/olmoearth_pretrain_dataset", repo_type="dataset")
    for folder in ["10_sentinel2_l2a_monthly", "10_sentinel1_monthly", "10_cdl", "10_worldcover", "10_worldcereal", "10_srtm"]:
        expected_count = sum(1 for f in files if f.startswith(folder + "/") and f.endswith(".tar"))
        local_count = int(shell(f"find {STAGING}/{folder} -type f -name '*.tar' 2>/dev/null | wc -l") or 0)
        mark = "✓" if expected_count and local_count == expected_count else ""
        print(f"{folder:<32} local={local_count:<5} hf={expected_count:<5} {mark}")
except Exception as e:
    print(f"跳过 HF shard 数检查: {e}")


In [ ]:
# 4.2 (可选)查特定 folder 的报错
FOLDER = "10_sentinel2_l2a_monthly"
!grep "\[$FOLDER\]" /root/download.log | tail -20

In [ ]:
# 4.3 (可选)测当前实时速率
# 间隔 60 秒采样 staging size,算下载速度
import os, time

def total_size():
    p = "/root/autodl-tmp/staging"
    return sum(os.path.getsize(os.path.join(r, f))
               for r, _, fs in os.walk(p) for f in fs)

s1 = total_size()
print(f"t=0  {s1/2**30:.2f} GB")
time.sleep(60)
s2 = total_size()
delta = (s2 - s1) / 2**20
print(f"t=60 {s2/2**30:.2f} GB  (+{delta:.1f} MB,{delta/60:.2f} MB/s)")

---
## 5. (可选)上传阿里云盘归档

**只在 §4 验证 ≥95% 完整后跑。** 上传是冷归档备份,**训练用数据盘本地副本,不靠云盘**。

阿里云盘**支持本地挂载**(macOS / Windows 客户端),传完后 Mac Finder 直接看到 ckpt + 日志。

In [ ]:
# 5.1 装 aliyunpan CLI(tickstep/aliyunpan,Go 实现)
!cd /root && wget -q https://github.com/tickstep/aliyunpan/releases/latest/download/aliyunpan-v0.3.7-linux-amd64.zip -O aliyunpan.zip
!cd /root && unzip -o -q aliyunpan.zip
!ln -sf "$(ls -d /root/aliyunpan-v*-linux-amd64)/aliyunpan" /usr/local/bin/alipan
!alipan --version

**5.2 登录阿里云盘** —— 在终端跑 `alipan login`(交互输入,notebook 跑不动)。

**拿 refresh token:**
1. 浏览器登录 https://www.alipan.com/
2. F12 → Application → Local Storage → 找 `token`
3. 复制 JSON 里的 `refresh_token` 字段值
4. 终端 `alipan login -RefreshToken=<粘这里>`

登录后回来跑 §5.3。

In [ ]:
# 5.3 启动上传到阿里云盘(后台,~30-50 hr / 16-26 hr 超会)
import subprocess, os

upload_cmd = '''#!/bin/bash
for folder in 10_sentinel2_l2a_monthly 10_sentinel1_monthly \
              10_srtm 10_worldcereal 10_worldcover 10_cdl; do
  echo "=== $(date) UPLOAD START $folder ==="
  alipan upload /root/autodl-tmp/staging/$folder \
    /OlmoEarth/$folder --np 16
  echo "=== $(date) UPLOAD DONE $folder ==="
done

echo "=== $(date) UPLOAD START vjepa ==="
alipan upload /root/autodl-tmp/staging/vjepa /VJEPA2/ckpts --np 4
echo "=== $(date) ALL UPLOADS DONE ==="
'''

with open("/root/upload_alipan.sh", "w") as f:
    f.write(upload_cmd)

subprocess.Popen(
    "nohup bash /root/upload_alipan.sh > /root/upload.log 2>&1 &",
    shell=True,
    preexec_fn=os.setpgrp,
)
print("阿里云盘上传启动")
print("日志: /root/upload.log")

In [ ]:
# 5.4 监控上传进度
!tail -30 /root/upload.log
print("\n=== 阿里云盘内容 ===")
!alipan ls /OlmoEarth/

---
## 6. 完成后,切换到 GPU 实例训练

**数据准备完成的判定:** §4.1 验证表 S2 + S1 + 小数据全部显示大小 OK,且无 incomplete/lock/tmp 文件;若 HF shard 数检查可用,local 数量应等于 hf 数量。

**切到 4× Pro 6000:**
1. AutoDL Console → 当前 instance → 关机
2. 同一 instance 选 4× Pro 6000 → 开机
3. 数据盘自动挂载(数据还在)
4. SSH 进入,启动训练:

```bash
cd /root/code/VJEPA2-Sentinel2-FineTune
git pull

# 改 yaml:source: local + tar_path 指向已下载数据
# vjepa2/configs/finetune/vitl16/olmoearth-256px-12f.yaml:
#   source: local
#   tar_path: "/root/autodl-tmp/staging/10_sentinel2_l2a_monthly/*.tar"
#   pretrained_checkpoint: /root/autodl-tmp/staging/vjepa/vjepa2_1_vitl_dist_vitG_384.pt

torchrun --nproc_per_node=4 finetune_main.py   --config vjepa2/configs/finetune/vitl16/olmoearth-256px-12f.yaml
```

**SAR-only smoke test 配置:**

```yaml
olmoearth:
  source: local
  tar_path: "/root/autodl-tmp/staging/10_sentinel1_monthly/*.tar"
  n_bands_per_timestep: 2
  norm: "sar_db"
  dn_scale: 1.0
model:
  in_chans: 2
```

**先跑 diagnostics_m0.py 验证(~1 hr,~¥10):**

```bash
python diagnostics_m0.py   --config vjepa2/configs/finetune/vitl16/olmoearth-256px-12f.yaml   --checkpoint /root/autodl-tmp/staging/vjepa/vjepa2_1_vitl_dist_vitG_384.pt   --data_dir /root/autodl-tmp/staging
```

**本地挂载阿里云盘:** Mac 装阿里云盘官方 app → Finder 直接看 ckpt + 日志,不用每次 SSH 拉。
